In [1]:
from TIEModel import TIEModel
import torch, math
from torch.optim import AdamW
from transformers import get_cosine_schedule_with_warmup
from TIEUtils import collator, TemporalDataset
from torch.utils.data import DataLoader
from globals import ID2LABEL_EVNER, ID2LABEL_EE, LABEL2ID_EVNER, LABEL2ID_EE

d:\GeoTKG\venv\Lib\site-packages\huggingface_hub\file_download.py:945: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


In [2]:
label2id_ner = LABEL2ID_EVNER
id2label_ner = ID2LABEL_EVNER
label2id_ee = LABEL2ID_EE
id2label_ee = ID2LABEL_EE
cleandata_path = "D:\\GeoTKG\\cleandata\\tie\\"
def collate_fn(examples):
    return collator(examples, label2id_ner=label2id_ner, label2id_ee=label2id_ee)
train = TemporalDataset(cleandata_path + "train.json")
eval = TemporalDataset(cleandata_path + "eval.json")
train_loader = DataLoader(train, batch_size=16, shuffle=True, collate_fn=collate_fn)
eval_loader = DataLoader(eval, batch_size=16, shuffle=False, collate_fn=collate_fn)

In [3]:
NUM_EPOCHS = 50
ENC_LR = 5e-5
NONENC_LR = 1e-3
WARMUP_EPOCHS = 10
LOGGING_EPOCHS = 1
BASE_ENC_MODEL = "roberta-base"
HEADS = 6
WEIGHT_DECAY = 0.01

In [4]:
device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')

model = TIEModel(base=BASE_ENC_MODEL, num_ner=len(label2id_ner), ee_labels=len(label2id_ee), heads=HEADS).to(device)

for p in model.enc.parameters(): p.requires_grad = False

optimizer = AdamW([
        {"params": [p for n,p in model.named_parameters() if n.startswith("enc.")], "lr": ENC_LR},
        {"params": [p for n,p in model.named_parameters() if not n.startswith("enc.")], "lr": NONENC_LR},
    ], weight_decay=0.01)

steps_per_epoch = len(train_loader)
num_train_steps = steps_per_epoch * NUM_EPOCHS
num_warmup_steps = steps_per_epoch * WARMUP_EPOCHS

sched = get_cosine_schedule_with_warmup(optimizer, num_warmup_steps, num_train_steps)

scaler = torch.amp.GradScaler(enabled=(device.type=='cuda'))

def make_optim(unfrozen: bool):
    groups = []
    if unfrozen:
        groups.append({"params": model.enc.parameters(), "lr": ENC_LR})
    else:
        # keep enc group empty or skip entirely; either is fine
        pass
    nonenc = [p for n, p in model.named_parameters() if not n.startswith("enc.")]
    groups.append({"params": nonenc, "lr": NONENC_LR})
    return AdamW(groups, weight_decay=WEIGHT_DECAY)

Some weights of the model checkpoint at roberta-base were not used when initializing RobertaModel: ['lm_head.layer_norm.weight', 'lm_head.dense.weight', 'lm_head.bias', 'lm_head.layer_norm.bias', 'lm_head.dense.bias']
- This IS expected if you are initializing RobertaModel from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing RobertaModel from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Some weights of RobertaModel were not initialized from the model checkpoint at roberta-base and are newly initialized: ['roberta.pooler.dense.weight', 'roberta.pooler.dense.bias']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [5]:
global_step = 0
history = {#  Training
           "loss": [], "ner_loss":[], "ptr_loss":[], "ee_loss":[],
           #  Evaluation
           "ner_f1": [], "ptr_acc": [], "ee_f1": [], "eval_loss":[], "ner_eval_loss": [], "ptr_eval_loss": [], "ee_eval_loss": []}
for epoch in range(NUM_EPOCHS):
    model.train()
    optimizer.zero_grad(set_to_none=True)
    loss_sum = 0
    ner_loss_sum = 0
    ptr_loss_sum = 0
    ee_loss_sum = 0
    ration = 0
    for step, batch in enumerate(train_loader):
        batch = {k: (v.to(device) if torch.is_tensor(v) else v) for k, v in batch.items()}

        ctx = (torch.autocast(device_type='cuda', dtype=torch.float16))
        with ctx:
            out = model(
                input_ids=batch["input_ids"],
                attention_mask=batch["attention_mask"],
                ev_starts=batch["ev_starts"], ev_ends=batch["ev_ends"], ev_mask=batch["ev_mask"], e_sent_ids=batch["e_sent_ids"],
                ti_starts=batch["ti_starts"], ti_ends=batch["ti_ends"], ti_mask=batch["ti_mask"], t_sent_ids=batch["t_sent_ids"],
                ner_gold_labels=batch["ner_labels"],
                ev_ti_gold=batch["ev_ti_gold"],
                ee_rel_gold=batch["ee_triples"],
                ee_mask=batch["ee_mask"],
            )
            loss = out["loss"]
            loss_sum += loss.item()
            ner_loss_sum += out["ner_loss"].item()
            ptr_loss_sum += out["ptr_loss"].item()
            ee_loss_sum += out["ee_loss"].item()
            ration += out['none_r']

        scaler.scale(loss).backward()

        scaler.unscale_(optimizer)
        #torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        prev_scale = scaler.get_scale()
        scaler.step(optimizer)
        scaler.update()
        sched.step() if scaler.get_scale() <= prev_scale else None
        optimizer.zero_grad(set_to_none=True)
        global_step += 1

    print(f"Epoch {epoch+1} done. Avg Loss: {loss_sum / steps_per_epoch:.4f}. Avg None Ratio {ration/steps_per_epoch:.4f}")
 
    # ---- unfreeze after warmup epochs ----
    if epoch + 1 == WARMUP_EPOCHS:
        for p in model.enc.parameters():
            p.requires_grad = True
        # rebuild optimizer & scheduler for the remaining steps
        optimizer = make_optim(unfrozen=True)
        remaining_steps = steps_per_epoch * (NUM_EPOCHS - (epoch + 1))
        warmup_rem = 0
        sched = get_cosine_schedule_with_warmup(optimizer, warmup_rem, remaining_steps)

    # ---- validation ----
    if (epoch + 1) % LOGGING_EPOCHS == 0:
        model.eval()
        with torch.no_grad():
            batch_evaluation = model.evaluate_dataloader(eval_loader, id2label_ner, id2label_ee)
            history["ner_f1"].append(batch_evaluation["ner_f1"])
            history["ptr_acc"].append(batch_evaluation["ptr_acc"])
            history["ee_f1"].append(batch_evaluation["ee_f1"])
            history["eval_loss"].append(batch_evaluation["eval_loss"])
            history["ner_eval_loss"].append(batch_evaluation["ner_loss"])
            history["ptr_eval_loss"].append(batch_evaluation["ptr_loss"])
            history["ee_eval_loss"].append(batch_evaluation["ee_loss"])
            history["loss"].append(loss_sum / steps_per_epoch)
            history["ner_loss"].append(ner_loss_sum / steps_per_epoch)
            history["ptr_loss"].append(ptr_loss_sum / steps_per_epoch)
            history["ee_loss"].append(ee_loss_sum / steps_per_epoch)
        model.save(f"results/tie_model/tie_model_epoch{epoch+1}.pt")
        print(f"EPOCH{epoch+1} \n NER F1={batch_evaluation['ner_f1']:.4f},  PTR={batch_evaluation['ptr_acc']:.4f},  EE F1={batch_evaluation['ee_f1']:.4f} \n Train Loss={loss_sum / steps_per_epoch:.4f},  Eval Loss={batch_evaluation['eval_loss']:.4f} \nNER Eval Loss={batch_evaluation['ner_loss']:.4f}, NER Train Loss={ner_loss_sum / steps_per_epoch:.4f},\nPTR Eval Loss={batch_evaluation['ptr_loss']:.4f}, PTR Train Loss={ptr_loss_sum / steps_per_epoch:.4f}\nEE Eval Loss={batch_evaluation['ee_loss']:.4f},EE Train Loss={ee_loss_sum / steps_per_epoch:.4f}")

d:\GeoTKG\venv\Lib\site-packages\torch\optim\lr_scheduler.py:182: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  warnings.warn(


Epoch 1 done. Avg Loss: 2.0438. Avg None Ratio 0.0199
Real correct PTR:  0.032761026512164046
None correct PTR:  0.04252866589819814


d:\GeoTKG\venv\Lib\site-packages\seqeval\metrics\v1.py:57: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


              precision    recall  f1-score   support

        DATE     0.0000    0.0000    0.0000      2459
    DURATION     0.0000    0.0000    0.0000       448
       EVENT     0.5093    0.1165    0.1896     16470
         SET     0.0000    0.0000    0.0000        11
        TIME     0.0000    0.0000    0.0000        70

   micro avg     0.4723    0.0986    0.1632     19458
   macro avg     0.1019    0.0233    0.0379     19458
weighted avg     0.4311    0.0986    0.1605     19458

              precision    recall  f1-score   support

       AFTER     0.6959    0.5132    0.5907      7192
      BEFORE     0.4672    0.7924    0.5878      5790
    CONTAINS     0.8008    0.6681    0.7284      6948
      DURING     0.0000    0.0000    0.0000       375
      EQUALS     0.5850    0.2367    0.3370      1686
    IDENTITY     0.7701    0.9200    0.8384      3823
    OVERLAPS     0.0000    0.0000    0.0000       357

    accuracy                         0.6433     26171
   macro avg     0.4741

d:\GeoTKG\venv\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
d:\GeoTKG\venv\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
d:\GeoTKG\venv\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


...

d:\GeoTKG\venv\Lib\site-packages\seqeval\metrics\v1.py:57: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


              precision    recall  f1-score   support

        DATE     0.6448    0.7471    0.6922      2459
    DURATION     0.1636    0.1384    0.1499       448
       EVENT     0.7476    0.6277    0.6825     16470
         SET     0.0000    0.0000    0.0000        11
        TIME     0.2143    0.1286    0.1607        70

   micro avg     0.7162    0.6294    0.6700     19458
   macro avg     0.3541    0.3284    0.3371     19458
weighted avg     0.7188    0.6294    0.6692     19458

              precision    recall  f1-score   support

       AFTER     0.7513    0.7900    0.7702      7192
      BEFORE     0.6500    0.7487    0.6959      5790
    CONTAINS     0.7896    0.7454    0.7669      6948
      DURING     0.7907    0.0907    0.1627       375
      EQUALS     0.6236    0.3517    0.4498      1686
    IDENTITY     0.8436    0.9527    0.8948      3823
    OVERLAPS     0.6232    0.1204    0.2019       357

    accuracy                         0.7454     26171
   macro avg     0.7246

d:\GeoTKG\venv\Lib\site-packages\seqeval\metrics\v1.py:57: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


              precision    recall  f1-score   support

        DATE     0.6625    0.7584    0.7072      2459
    DURATION     0.2058    0.1897    0.1974       448
       EVENT     0.7480    0.6392    0.6894     16470
         SET     0.0000    0.0000    0.0000        11
        TIME     0.3617    0.2429    0.2906        70

   micro avg     0.7202    0.6422    0.6789     19458
   macro avg     0.3956    0.3661    0.3769     19458
weighted avg     0.7229    0.6422    0.6785     19458

              precision    recall  f1-score   support

       AFTER     0.7692    0.7970    0.7828      7192
      BEFORE     0.7703    0.6297    0.6930      5790
    CONTAINS     0.7284    0.8008    0.7629      6948
      DURING     0.5979    0.1547    0.2458       375
      EQUALS     0.5424    0.4063    0.4646      1686
    IDENTITY     0.7916    0.9717    0.8725      3823
    OVERLAPS     0.2857    0.2353    0.2581       357

    accuracy                         0.7445     26171
   macro avg     0.6408

KeyboardInterrupt: 

In [ ]:
import json
with open("results/tie_model/history.json", "w") as f:
    json.dump(history, f, indent=2)

In [ ]:
import matplotlib.pyplot as plt

x = list(range(5, 21, 5))

plt.plot(x, history["ner_f1"], label="NER F1")
plt.plot(x, history["ptr_acc"], label="Pointer Accuracy")
plt.plot(x, history["ee_f1"], label="EE F1")
plt.xlabel("Epoch")
plt.ylabel("Score")
plt.title("Evaluation Metrics Over Time")
plt.legend()
plt.show()

In [ ]:
plt.plot(x, history["loss"], label="Loss")
plt.plot(x, history["ner_loss"], label="NER Loss")
plt.plot(x, history["ee_loss"], label="EE Loss")
plt.plot(x, history["ptr_loss"], label="PTR Loss")
plt.legend()
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training Loss")
plt.show()

In [ ]:
plt.plot(x, history["eval_loss"], label="Loss")
plt.plot(x, history["ner_eval_loss"], label="NER Loss")
plt.plot(x, history["ee_eval_loss"], label="EE Loss")
plt.plot(x, history["ptr_eval_loss"], label="PTR Loss")
plt.legend()
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Evaluation Loss")
plt.show()

In [ ]:
plt.plot(x, history["eval_loss"], label="Eval Loss")
plt.plot(x, history["loss"], label="Train Loss")
plt.legend()
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Compare Loss")
plt.show()